In [ ]:
import logging
import csv
from typing import Any, Dict, List
import json
import json
import os
import sys
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv
import difflib

from kinexon_handball_api.handball import HandballAPI

In [ ]:
load_dotenv()  # Load environment variables from .env file if present


In [ ]:
# connect duckdb
import duckdb
con = duckdb.connect(database='../data/mydb2024-25.duckdb', read_only=False)

In [ ]:
api = HandballAPI(
    base_url=os.getenv(
        "ENDPOINT_KINEXON_SESSION", "https://hbl-cloud.kinexon.com/api"
    ),
    api_key=os.getenv("API_KEY_KINEXON", "your_api_key_here"),
    username_basic=os.getenv("USERNAME_KINEXON_SESSION", "your_username_here"),
    password_basic=os.getenv("PASSWORD_KINEXON_SESSION", "your_password_here"),
    username_main=os.getenv("USERNAME_KINEXON_MAIN", "your_username_here"),
    password_main=os.getenv("PASSWORD_KINEXON_MAIN", "your_password_here"),
    endpoint_session=os.getenv(
        "ENDPOINT_KINEXON_SESSION",
        "https://hbl-cloud.kinexon.com/api/session",
    ),
    endpoint_main=os.getenv(
        "ENDPOINT_KINEXON_MAIN",
        "https://hbl-cloud.kinexon.com/api",
    ),
    timeout=10000,
)

In [ ]:
avail = api.get_available_metrics_and_events()
# print("Available metrics and events:", len(avail))

ids_team = api.fetch_team_ids("2024-25")

print("Team IDs:", ids_team)

In [ ]:
# select * from table teams
df_teams_sportradar = con.execute("SELECT * FROM teams").df()
print("teams")
display(df_teams_sportradar.head())

# select * from table fixtures

df_fixtures_sportradar = con.execute("SELECT * FROM fixtures").df()
print("Fixtures")
display(df_fixtures_sportradar.head())

## Iterate of Sportradar fixtures and download the session id from kinexon

In [ ]:
# # https://hbl-cloud.kinexon.com/export/check-session-raw-bin-file-exists
# # body: { "sessionId": "string" }
# def check_bin_file_exists(session_id: str) -> bool:
#     url = f"{api.endpoint_session}/export/check-session-raw-bin-file-exists"
#     body = {"sessionId": session_id}
#     response = api.make_custom_request(
#         method="POST",
#         url=url,
#         json=body
#     )
#     if response and response.content and "true" in str(response.content):
#         return True
#     return False

# import os
# import re
# import requests
# from requests.auth import HTTPBasicAuth
# from typing import Optional, Tuple

# FILENAME_RE = re.compile(r'filename\*?=(?:UTF-8\'\')?"?([^";]+)"?')
# from urllib.parse import unquote, quote
# def parse_urlencoded_json(payload_urlencoded: str) -> dict:
#     """Decode %7B...%7D into a Python dict."""
#     return json.loads(unquote(payload_urlencoded))

# def encode_params(params: dict) -> str:
#     """Compact JSON then percent-encode for application/x-www-form-urlencoded."""
#     return quote(json.dumps(params, separators=(",", ":")))

# def _filename_from_headers(headers: requests.structures.CaseInsensitiveDict) -> Optional[str]:
#     cd = headers.get("Content-Disposition", "")
#     m = FILENAME_RE.search(cd)
#     return m.group(1) if m else None

# def download_kinexon_csv_for_session(
#     team_id: int,
#     session_id: str,
#     *,
#     params: Optional[dict] = None,          # pass a Python dict
#     payload_urlencoded: Optional[str] = None,  # or pass the %7B...%7D string directly
#     return_filename_and_bytes: bool = False
# ) -> Any:
#     """
#     POSTs to:
#       {endpoint_session}/teams/{team_id}/sessions/kinexon-session-{session_id}-export.csv
#     Body must be a raw percent-encoded JSON string with Content-Type: application/x-www-form-urlencoded.
#     """
#     if not (params or payload_urlencoded):
#         raise ValueError("Provide either 'params' or 'payload_urlencoded'.")

#     if params and payload_urlencoded:
#         raise ValueError("Provide only one of 'params' or 'payload_urlencoded', not both.")

#     data = payload_urlencoded or encode_params(params)  # ensure percent-encoded JSON

#     url = f"{api.endpoint_session}/teams/{team_id}/sessions/kinexon-session-{session_id}-export.csv"
#     headers = {
#         "Content-Type": "application/x-www-form-urlencoded",
#         "Accept": "text/csv, */*",
#     }

#     resp = api.make_custom_request(
#         method="POST",
#         url=url,
#         headers=headers,
#         data=data,          # IMPORTANT: raw string, not dict
#         stream=False,
#     )
#     csv_bytes = resp.content

#     if return_filename_and_bytes:
#         fname = _filename_from_headers(resp.headers) or f"kinexon-session-{session_id}.csv"
#         return fname.replace("/", "_"), csv_bytes
#     return csv_bytes





In [ ]:
import io
import gzip
import zipfile
import pandas as pd

# --- magic-byte sniffers ---
def _is_gzip(b: bytes) -> bool:
    # 1F 8B
    return len(b) >= 2 and b[0] == 0x1F and b[1] == 0x8B

def _is_zip(b: bytes) -> bool:
    # PK\x03\x04
    return len(b) >= 4 and b[:4] == b"PK\x03\x04"

def read_positions_payload(payload: bytes) -> pd.DataFrame:
    """
    Accepts raw bytes from the API. Detects gzip/zip and returns a DataFrame.
    Assumes UTF-8 CSV inside.
    """
    raw = payload
    if _is_gzip(raw):
        raw = gzip.decompress(raw)
        csv_bytes = raw
    elif _is_zip(raw):
        with zipfile.ZipFile(io.BytesIO(raw)) as zf:
            # pick the first CSV entry
            csv_name = next((n for n in zf.namelist() if n.lower().endswith(".csv")), None)
            if not csv_name:
                raise ValueError("ZIP archive does not contain a CSV file.")
            csv_bytes = zf.read(csv_name)
    else:
        # assume uncompressed CSV
        csv_bytes = raw

    # parse CSV
    return pd.read_csv(io.StringIO(csv_bytes.decode("utf-8-sig")), sep=";")


In [ ]:
import io
import concurrent.futures
from tqdm import tqdm


def process_session(row):
    session_id = row["session_id"]
    fixture_id = row["fixtureId"]
    # get team id from api
    ids_team = api.fetch_team_ids("2024-25")
    # ids_team look like: ids_team = [{'id': 2, 'name': 'HC Erlangen'}, {'id': 3, 'name': 'HSG Wetzlar'}, {'id': 5, 'name': 'TBV Lemgo Lippe'}, {'id': 6, 'name': 'MT Melsungen'}, {'id': 7, 'name': 'Rhein-Neckar Löwen'}, {'id': 8, 'name': 'TSV Hannover-Burgdorf'}, {'id': 9, 'name': 'SC DHFK Leipzig'}, {'id': 11, 'name': 'Frisch Auf! Göppingen'}, {'id': 12, 'name': 'SG Flensburg-Handewitt'}, {'id': 13, 'name': 'Füchse Berlin'}, {'id': 16, 'name': 'SC Magdeburg'}, {'id': 17, 'name': 'TVB Stuttgart'}, {'id': 18, 'name': 'THW Kiel'}, {'id': 23, 'name': 'HSV Hamburg'}, {'id': 32, 'name': 'VfL Gummersbach'}, {'id': 33, 'name': 'ThSV Eisenach'}, {'id': 35, 'name': 'SG BBM Bietigheim'}, {'id': 36, 'name': '1. VfL Potsdam'}]
    team_id = None
    for team in ids_team:
        if team["name"] == row["name_team_home"]:
            team_id = team["id"]
            break

    # Check if session_id is already in the database
    try:
        existing = con.execute(
            "SELECT COUNT(*) AS cnt FROM kinexon_positions WHERE session_id = ?", (session_id,)
        ).fetchone()
        if existing and existing[0] > 0:
            return {
                "session_id": session_id,
                "fixture_id": fixture_id,
                "records": 0,
                "status": "skipped (already in DB)",
            }  
    
    except duckdb.CatalogException:
        # Table does not exist yet
        pass

    try:
        print(f"Downloading positions for session_id: {session_id}, fixture_id: {fixture_id}, team_id: {team_id}")
        print("Gamedate info:", row.to_dict())

        response = api.download_positions_csv_via_custom(session_id=session_id, compress_output=True)

        if isinstance(response, bytes):
            df_positions = read_positions_payload(response)
        else:
            # if it returns an httpx.Response, use .content
            df_positions = read_positions_payload(response.content)

        df_positions["session_id"] = session_id
        df_positions["fixture_id"]  = fixture_id

        if len(df_positions) > 0:
            con.execute(
                "CREATE TABLE IF NOT EXISTS kinexon_positions AS SELECT * FROM df_positions WHERE 1=0"
            )
            con.execute(
                "INSERT INTO kinexon_positions SELECT * FROM df_positions"
            )

        return {
            "session_id": session_id,
            "fixture_id": fixture_id,
            "records": len(df_positions),
            "status": "ok",
        }
    except Exception as e:
        return {
            "session_id": session_id,
            "fixture_id": fixture_id,
            "records": 0,
            "status": f"error: {e}",
        }

In [ ]:
# # display existing kinexon_positions table, but with distinct session_ids showing complete rows
# df_existing_kinexon = con.execute("""
#     SELECT * FROM kinexon_positions 
#     WHERE session_id IN (
#         SELECT DISTINCT session_id FROM kinexon_positions LIMIT 5
#     )
#     ORDER BY session_id
# """).df()
# print("Existing Kinexon positions data:")
# display(df_existing_kinexon.head())

In [ ]:
# drop table kinexon_positions for fresh start
# con.execute("DROP TABLE IF EXISTS kinexon_positions")

In [ ]:
# # --- single-threaded execution ---
# results = []

# for _, row in tqdm(df_fixtures_sportradar.iterrows(), total=len(df_fixtures_sportradar)):
#     res = process_session(row)
#     results.append(res)
#     sid = res["session_id"]
#     status = res["status"]
#     if status == "ok":
#         print(f"[{sid}] ✓ inserted {res['records']} records")
#     else:
#         print(f"[{sid}] ✖ {status}")

# # Optional summary
# df_summary = pd.DataFrame(results)
# print("\n=== Summary ===")
# print(df_summary["status"].value_counts())
# print(f"Total records inserted: {df_summary['records'].sum()}")


In [ ]:

# --- multithreaded execution ---
MAX_WORKERS = 10
results = []

with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {
        executor.submit(process_session, row): row["session_id"]
        for _, row in df_fixtures_sportradar.iterrows()
    }

    for future in tqdm(concurrent.futures.as_completed(futures), total=len(futures)):
        res = future.result()
        results.append(res)
        sid = res["session_id"]
        status = res["status"]
        if status == "ok":
            print(f"[{sid}] ✓ inserted {res['records']} records")
        else:
            print(f"[{sid}] ✖ {status}")

# Optional summary
df_summary = pd.DataFrame(results)
print("\n=== Summary ===")
print(df_summary["status"].value_counts())
print(f"Total records inserted: {df_summary['records'].sum()}")


In [ ]:
# import io

# for index, row in df_fixtures_sportradar.iterrows():
#     session_id = row['session_id']
#     print(f"Processing session_id ID: {session_id}")
    
#     try:
#         response = api.download_positions_csv_via_custom(session_id=session_id)
        
#         if isinstance(response, bytes):
#             # Decode bytes to string and read as CSV
#             csv_string = response.decode('utf-8')
#             df_positions = pd.read_csv(io.StringIO(csv_string))
#             print(f"Downloaded {len(df_positions)} position records for session_id {session_id}")
            
#             # Add session_id to the DataFrame for tracking
#             df_positions['session_id'] = session_id
#             df_positions['fixture_id'] = row['fixtureId']  # Link to fixture
            
#             # Store in DuckDB following project patterns
#             if len(df_positions) > 0:
#                 con.execute("CREATE TABLE IF NOT EXISTS kinexon_positions AS SELECT * FROM df_positions WHERE 1=0")
#                 con.execute("INSERT INTO kinexon_positions SELECT * FROM df_positions")
#                 print(f"  Inserted {len(df_positions)} records into kinexon_positions table")
            
#         else:
#             print(f"  Unexpected response type: {type(response)}")
#             print(f"  Response content: {response}")
            
#     except Exception as e:
#         print(f"  Error processing session_id {session_id}: {str(e)}")
#         continue